In [ ]:
from __future__ import annotations

import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import wandb
from scipy.signal import find_peaks
from tqdm.autonotebook import tqdm


### File-parsing function

In [ ]:
def parse_file(df: pd.DataFrame, column_to_parse="file") -> pd.DataFrame:
    df["file_stem"] = df[column_to_parse].apply(lambda f: Path(f).stem.split(".")[0])
    df["file_stem"] = df["file_stem"].str.replace(
        "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_", ""
    )
    df["cbt_number"] = df["file_stem"].apply(lambda f: int(f.split("-")[1]) if f.lower().startswith("cbt") else None)
    video_types = ["Intro", "Passage", "More Info"]
    df["cbt_video_type"] = df["file_stem"].apply(
        lambda f: next((v for v in video_types if v.lower() in f.lower()), None)
    )

    # Video title, e.g. "God Creates The World"
    df["title"] = df["file_stem"].apply(
        lambda f: f.split("_")[1].strip().title() if f.lower().startswith("cbt") else None
    )
    df["title"] = df["title"].astype("string")
    return df

### Load Video Window-Embeddings


In [ ]:
# window_embeddings_folder = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\chronological_bible_translation_in_american_sign_language_119_introductions_and_passages")
# import shutil

# children = list(window_embeddings_folder.glob("*"))
# out_dir = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\hive_format")
# out_dir.mkdir(exist_ok=True, parents=True)
# for child in children:
#     child_name = child.name
#     print(child_name)
#     # child.rename(ch)
#     new_name = child_name.replace(
#         "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_", ""
#     )
#     new_name = child_name.replace(".pose-mediapipe.window_embeddings", "")
#     new_name = f"video_stem={new_name}"
#     print(new_name)
#     new_path = out_dir / new_name
#     print(new_path)
#     shutil.copytree(child, new_path)

In [ ]:

window_embeddings_folder = Path(r"C:\Users\Colin Leong\data\DBL\signclip_embeddings\window_embeddings\ase\hive_format")

In [ ]:
dataset = ds.dataset(window_embeddings_folder, format="parquet",partitioning="hive")
dataset.count_rows()
# df = ds.count_rows
dataset.schema

In [ ]:
window_embeddings_df = dataset.to_table().to_pandas()

In [ ]:
window_embeddings_df["file"] = window_embeddings_df["video_stem"].str.replace(
    "ase_chronological_bible_translation_in_american_sign_language_119_introductions_and_passages_cbt", "CBT"
)
window_embeddings_df["file"] = window_embeddings_df["file"].str.replace("passage", "Passage")

In [ ]:
window_embeddings_df = parse_file(window_embeddings_df)
window_embeddings_df["title"] = window_embeddings_df["title"].str.title()

In [ ]:
window_embeddings_df.head()

In [ ]:
window_embeddings_df["window_size_ms"].unique()


In [ ]:
window_embeddings_df["title"].unique()

In [ ]:
window_embeddings_500_ms = window_embeddings_df[window_embeddings_df["window_size_ms"] ==500]

### Load Ground Truth


In [ ]:
osb_glosses_version="1_4"
osb_glosses_dir = Path(r"C:\Users\Colin Leong\projects\SignBiblesDatasetPaper\Avodah_Annotations") / f"osb_glosses_v{osb_glosses_version}" /"osb_glosses"
osb_glosses_dfs = []

for truth_file in osb_glosses_dir.rglob("*.tsv"):
    truth_file_df = pd.read_csv(truth_file, delimiter="\t")
    # truth_file_df['video_title'] = truth_file.stem # alread included.
    truth_file_df = parse_file(truth_file_df)
    osb_glosses_dfs.append(truth_file_df)
osb_glosses_truth_df = pd.concat(osb_glosses_dfs)
osb_glosses_truth_df["window_midpoint_ms"] = osb_glosses_truth_df["midpoint_sec"] * 1000
osb_glosses_truth_df = osb_glosses_truth_df.rename(columns={"Sign":"Original Sign","ASL Citizen Gloss":"Sign"})
osb_glosses_truth_df.head()

In [ ]:
len(osb_glosses_truth_df)

In [ ]:
len(osb_glosses_truth_df["Sign"].unique())

### Load isolated embeddings

In [ ]:
isolated_sign_embeddings_dir = Path(r"C:\Users\Colin Leong\data\isolated_sign_embeddings\embeddings")
dataset = ds.dataset(isolated_sign_embeddings_dir, format="parquet",partitioning="hive")
ds_filter = ((ds.field("dataset")=="asl-citizen") & (ds.field("embedding_model")=="asl_finetune_checkpoint_best"))
isolated_sign_embeddings_df = dataset.to_table(filter=ds_filter).to_pandas()

In [ ]:
isolated_sign_embeddings_df.head()

### Load Transcripts

In [ ]:
import json

In [ ]:
transcripts_dir = Path(
    r"C:\Users\Colin Leong\data\DBL\webdataset_extracted_transcripts_only\transcripts\data\petabyte\cleong\data\DBL_Deaf_Bibles\webdataset_extracted\ase\chronological_bible_translation_in_american_sign_language_119_introductions_and_passages"
)


transcripts_collector = []
for transcript_file in transcripts_dir.glob("*.json"):
    print(transcript_file.name)
    with open(transcript_file, "r") as f:
        transcripts_for_file = json.load(f)
        print(transcripts_for_file)
        transcripts_for_file_df = pd.DataFrame.from_records(transcripts_for_file)
        transcripts_for_file_df["file"] = transcript_file.name
        transcripts_collector.append(transcripts_for_file_df)
transcripts_df = pd.concat(transcripts_collector)
transcripts_df = parse_file(transcripts_df)

In [ ]:
transcripts_df

### Pick a video and ground truth and search for glosses

In [ ]:

window_size_ms = 500
example_count = 25

# storage for results
results = []

for (cbt_number, cbt_video_type), file_gt_df in osb_glosses_truth_df.groupby(["cbt_number", "cbt_video_type"]):
    title = file_gt_df["title"][0]
    file_window_embeddings_df = window_embeddings_df[
        (window_embeddings_df["cbt_number"] == cbt_number)
        & (window_embeddings_df["cbt_video_type"] == cbt_video_type)
        & (window_embeddings_df["window_size_ms"] == window_size_ms)
    ]

    if file_window_embeddings_df.empty:
        continue

    file_window_embeddings_array = np.vstack(file_window_embeddings_df["embedding"])

    for sign in file_gt_df["Sign"].unique():
        sign_df = isolated_sign_embeddings_df[isolated_sign_embeddings_df["GLOSS"] == sign]

        if len(sign_df) == 0:
            continue

        # sample consistently for comparability
        n_sample = min(len(sign_df), example_count)
        sign_df.sample(n=example_count, random_state=0)

        sign_df_embeddings = np.vstack(sign_df["embedding"])
        sign_scores = sign_df_embeddings @ file_window_embeddings_array.T  # (n_examples, n_windows)

        # aggregate across the sampled examples
        mean_sign_scores = sign_scores.mean(axis=0)  # (n_windows,)

        # collect results
        for idx, score in enumerate(mean_sign_scores):
            results.append(
                {
                    "cbt_number": cbt_number,
                    "cbt_video_type": cbt_video_type,
                    "Sign": sign,
                    "window_index": idx,
                    "window_midpoint_ms": file_window_embeddings_df["window_midpoint_ms"].iloc[idx],
                    "score": score,
                    "title": title,
                }
            )

# turn into DataFrame for plotting
results_df = pd.DataFrame(results)


#### Plot peaks

In [ ]:
import plotly.express as px

# group by cbt_number + cbt_video_type
for (cbt_number, cbt_video_type), group_df in results_df.groupby(["cbt_number", "cbt_video_type"]):
    assert len(group_df["title"].unique()) == 1
    video_title = group_df["title"].unique()[0]
    fig = px.line(
        group_df,
        x="window_midpoint_ms",
        y="score",
        color="Sign",
        title=f"Mean Sign Scores — {video_title}",
    )
    fig.show()


### apply find_peaks

In [ ]:
def find_peaks_in_df(
    df: pd.DataFrame,
    group_col="Sign",
    x_col="window_midpoint_ms",
    y_col="score",
    prominence=0.6,
    height=None,
    width=None,  # in number of samples. TODO: set based on step size?
    distance=None,
) -> pd.DataFrame:
    """
    Find peaks in each group and return a DataFrame of peak positions.

    Returns:
    - DataFrame with columns [group_col, x_col, y_col]

    """
    peak_rows = []
    for query_id, group in df.groupby(group_col):
        group = group.sort_values(x_col)
        x = group[x_col].values
        y = group[y_col].values

        # TODO: box plots of properties to investigate
        peaks, properties = find_peaks(y, prominence=prominence, height=height, width=width, distance=distance)
        for i in peaks:
            peak_rows.append(
                {
                    group_col: query_id,
                    x_col: x[i],
                    y_col: y[i],
                }
            )
    return pd.DataFrame(peak_rows, columns=[group_col, x_col, y_col])


### Match Predictions

In [ ]:
def match_predictions(
    preds_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    window: int = 1000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Match predictions to ground truth within a ±window (in ms).
    Loops over ground-truth entries to ensure FN are counted correctly even if
    preds_df is empty.
    """
    # At the start of match_predictions
    n_null_preds = preds_df["Sign"].isna().sum()
    n_null_gt = gt_df["Sign"].isna().sum()

    if n_null_preds > 0:
        print(f"Warning: Dropping {n_null_preds} predictions with null Sign")
    if n_null_gt > 0:
        print(f"Warning: Dropping {n_null_gt} ground truth entries with null Sign")

    preds_df = preds_df[preds_df["Sign"].notna()].copy()
    gt_df = gt_df[gt_df["Sign"].notna()].copy()

    preds_out = preds_df.copy()
    preds_out["correct"] = False
    gt_out = gt_df.copy()
    gt_out["found"] = False

    # Track which predictions have been matched to avoid double counting
    matched_preds = set()
    # Loop over GT rows
    for gt_idx, gt_row in gt_df.iterrows():
        gt_sign = gt_row["Sign"]
        gt_ts = gt_row["window_midpoint_ms"]
        # find matching prediction(s) within window
        candidates = preds_out[
            (preds_out["Sign"] == gt_sign)
            & (~preds_out.index.isin(matched_preds))
            & (abs(preds_out["window_midpoint_ms"] - gt_ts) <= window)
        ]
        if not candidates.empty:
            pred_idx = candidates.index[0]  # match first candidate
            preds_out.at[pred_idx, "correct"] = True
            gt_out.at[gt_idx, "found"] = True
            matched_preds.add(pred_idx)
    # --- per-sign metrics ---
    per_sign_records = []
    all_signs = sorted(set(gt_df["Sign"]).union(preds_df.get("Sign", [])))
    for sign in all_signs:
        gt_sign = gt_out[gt_out["Sign"] == sign]
        preds_sign = preds_out[preds_out["Sign"] == sign]

        # Count TP from ground truth side (each GT can only be found once)
        tp = int(gt_sign["found"].sum())
        # FP = predictions that weren't correct
        fp = int((~preds_sign["correct"]).sum())
        # FN = ground truth that wasn't found
        fn = int((~gt_sign["found"]).sum())

        prec = tp / len(preds_sign) if len(preds_sign) > 0 else float("nan")
        rec = tp / len(gt_sign) if len(gt_sign) > 0 else float("nan")
        f1_sign = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        per_sign_records.append(
            {
                "Sign": sign,
                "precision": prec,
                "recall": rec,
                "f1": f1_sign,
                "true_positives": tp,
                "false_positives": fp,
                "false_negatives": fn,
                "n_preds": len(preds_sign),
                "n_gt": len(gt_sign),
            }
        )
    per_sign_df = pd.DataFrame.from_records(per_sign_records).set_index("Sign")
    return preds_out, gt_out, per_sign_df


def aggregate_metrics(per_sign_df: pd.DataFrame) -> dict[str, float]:
    """Aggregate overall precision/recall/F1 from a per-sign stats DataFrame."""
    tp = int(per_sign_df["true_positives"].sum())
    fp = int(per_sign_df["false_positives"].sum())
    fn = int(per_sign_df["false_negatives"].sum())
    n_preds = int(per_sign_df["n_preds"].sum())
    n_gt = int(per_sign_df["n_gt"].sum())

    precision = tp / n_preds if n_preds > 0 else float("nan")
    recall = tp / n_gt if n_gt > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else float("nan")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "n_preds": n_preds,
        "n_gt": n_gt,
    }


In [ ]:
def compute_micro_metrics(df: pd.DataFrame) -> dict[str, float]:
    """Compute micro-averaged precision, recall, f1 from per-video metrics DataFrame."""
    required_cols = {"true_positives", "false_positives", "false_negatives"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns {required_cols}")

    total_tp = int(df["true_positives"].sum())
    total_fp = int(df["false_positives"].sum())
    total_fn = int(df["false_negatives"].sum())

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"micro_precision": precision, "micro_recall": recall, "micro_f1": f1}


# WandB Sweep

In [ ]:
# Filter out null signs from ground truth at the start
osb_glosses_truth_df = osb_glosses_truth_df[osb_glosses_truth_df["Sign"].notna()].copy()
n_null_signs = len(osb_glosses_truth_df) - len(osb_glosses_truth_df[osb_glosses_truth_df["Sign"].notna()])
if n_null_signs > 0:
    print(f"Filtered out {n_null_signs} ground truth entries with null Sign")


# --- hyperparameter grids ---
# height_multipliers = [1.0, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.5]
height_multipliers = list(np.arange(1.15, 1.3, 0.01))
random.shuffle(height_multipliers)

print(height_multipliers)
# prominences = [0.5, 0.6, 0.7, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 10.0, 100.0, 1000.0]
prominences = list(np.arange(2.0, 8.0, 0.1))
random.shuffle(prominences)
# widths = [None, 50, 100, 200, 500, 1000, 10000]
# widths = [None, 1, 10, 100, 1000]
widths = [None, 1, 2]
random.shuffle(widths)
distances = [
    1,
    # None,
    2,
    3,
    4,
    5,
    10,
    100,
]  # Required minimal horizontal distance (>= 1) in samples between neighbouring peaks. Smaller peaks are removed first until the condition is fulfilled for all remaining peaks.
random.shuffle(distances)
match_window_ms = 1000
# example_counts = [5, 10, 15, 25]
example_counts = list(range(1, 29))
random.shuffle(example_counts)
thresholdings = ["all-gloss", "per-gloss"]
random.shuffle(thresholdings)

# all available embedding window sizes
available_window_sizes = sorted(window_embeddings_df["window_size_ms"].unique())

# random.shuffle(available_window_sizes)

runs_per_window_size = 100

for embedding_window_ms in tqdm(available_window_sizes, desc="Embedding window sweep"):
    # --- generate results_df for this embedding window ---
    results = []
    for (cbt_number, cbt_video_type), file_gt_df in osb_glosses_truth_df.groupby(["cbt_number", "cbt_video_type"]):
        title = file_gt_df["title"].iloc[0]
        file_window_embeddings_df = window_embeddings_df[
            (window_embeddings_df["cbt_number"] == cbt_number)
            & (window_embeddings_df["cbt_video_type"] == cbt_video_type)
            & (window_embeddings_df["window_size_ms"] == embedding_window_ms)
        ]
        if file_window_embeddings_df.empty:
            continue

        file_window_embeddings_array = np.vstack(file_window_embeddings_df["embedding"])

        for sign in file_gt_df["Sign"].unique():
            sign_df = isolated_sign_embeddings_df[isolated_sign_embeddings_df["GLOSS"] == sign]
            if sign_df.empty:
                continue

            n_sample = min(len(sign_df), example_count)
            sign_df = sign_df.sample(n=n_sample, random_state=0)
            sign_embeddings = np.vstack(sign_df["embedding"])
            sign_scores = sign_embeddings @ file_window_embeddings_array.T
            mean_scores = sign_scores.mean(axis=0)

            for idx, score in enumerate(mean_scores):
                results.append(
                    {
                        "cbt_number": cbt_number,
                        "cbt_video_type": cbt_video_type,
                        "Sign": sign,
                        "window_index": idx,
                        "window_midpoint_ms": file_window_embeddings_df["window_midpoint_ms"].iloc[idx],
                        "score": score,
                        "title": title,
                    }
                )

    results_df = pd.DataFrame(results)
    if results_df.empty:
        continue

    # --- sweep over hyperparameters, including thresholding ---
    hyperparam_combos = list(
        itertools.product(height_multipliers, prominences, widths, example_counts, thresholdings, distances)
    )

    random.shuffle(hyperparam_combos)
    hyperparam_combos = hyperparam_combos[:runs_per_window_size]

    for height_multiplier, prominence, width, example_count, thresholding, distance in tqdm(
        hyperparam_combos, desc=f"Threshold sweep for {embedding_window_ms}ms"
    ):
        print(height_multiplier, prominence, width, example_count, thresholding, distance)
        run = wandb.init(
            project="sign-bibles-matching",
            config={
                "embedding_window_ms": embedding_window_ms,
                "height_multiplier": height_multiplier,
                "prominence": prominence,
                "width": width,
                "match_window_ms": match_window_ms,
                "example_count": example_count,
                "thresholding": thresholding,
                "distance": distance,
            },
            reinit="create_new",
            tags=["metrics_v3", f"osb_glosses_v{osb_glosses_version}"],
        )

        combo_video_metrics = []
        combo_gloss_metrics = []

        for (cbt_number, cbt_video_type), group_df in results_df.groupby(["cbt_number", "cbt_video_type"]):
            file_gt_df = osb_glosses_truth_df[
                (osb_glosses_truth_df["cbt_number"] == cbt_number)
                & (osb_glosses_truth_df["cbt_video_type"] == cbt_video_type)
            ]

            if thresholding == "all-gloss":
                group_mean = group_df["score"].mean()
                peaks_df = find_peaks_in_df(
                    group_df,
                    height=group_mean * height_multiplier,
                    prominence=prominence,
                    width=width,
                    distance=distance,
                )

            elif thresholding == "per-gloss":
                per_gloss_collector = []
                for sign, sign_results_df in group_df.groupby("Sign"):
                    sign_mean = sign_results_df["score"].mean()
                    sign_peaks = find_peaks_in_df(
                        sign_results_df,
                        height=sign_mean * height_multiplier,
                        prominence=prominence,
                        width=width,
                        distance=distance,
                    )
                    per_gloss_collector.append(sign_peaks)
                peaks_df = (
                    pd.concat(per_gloss_collector) if per_gloss_collector else pd.DataFrame(columns=group_df.columns)
                )

            preds_out, gt_out, per_sign_df = match_predictions(peaks_df, file_gt_df, window=match_window_ms)
            video_metrics = aggregate_metrics(per_sign_df)
            video_metrics.update(
                {
                    "cbt_number": cbt_number,
                    "cbt_video_type": cbt_video_type,
                    "thresholding": thresholding,
                }
            )
            combo_video_metrics.append(video_metrics)

            gloss_df = per_sign_df.reset_index().assign(
                cbt_number=cbt_number,
                cbt_video_type=cbt_video_type,
                thresholding=thresholding,
            )
            combo_gloss_metrics.append(gloss_df)

        video_metrics_df = pd.DataFrame(combo_video_metrics)
        gloss_metrics_df = pd.concat(combo_gloss_metrics, ignore_index=True)

        run.log(compute_micro_metrics(gloss_metrics_df))
        run.log({"video_metrics": wandb.Table(dataframe=video_metrics_df)})
        run.log({"gloss_metrics": wandb.Table(dataframe=gloss_metrics_df)})

        # Group by Sign and sum the count columns
        grouped_metrics = (
            gloss_metrics_df.groupby("Sign")
            .agg({
                "true_positives": "sum",
                "false_positives": "sum",
                "false_negatives": "sum",
                "n_preds": "sum",
                "n_gt": "sum",
            })
            .reset_index()
        )

        # Compute micro-metrics for each Sign, keeping counts
        sign_micro_metrics = []
        for _, row in grouped_metrics.iterrows():
            counts_df = pd.DataFrame([row])
            metrics = compute_micro_metrics(counts_df)
            metrics.update({
                "Sign": row["Sign"],
                "true_positives": row["true_positives"],
                "false_positives": row["false_positives"],
                "false_negatives": row["false_negatives"],
                "n_preds": row["n_preds"],
                "n_gt": row["n_gt"],
            })
            sign_micro_metrics.append(metrics)

        # Convert to DataFrame for logging
        sign_micro_df = pd.DataFrame(sign_micro_metrics)

        # Reorder columns for readability
        cols = [
            "Sign",
            "precision",
            "recall",
            "f1",
            "true_positives",
            "false_positives",
            "false_negatives",
            "n_preds",
            "n_gt",
        ]
        sign_micro_df = sign_micro_df[cols]

        # Log to W&B
        run.log({"grouped_sign_metrics": wandb.Table(dataframe=sign_micro_df)})

        run.finish()
